# vgra-yolo: evaluation session (I1 + I3 + I4 + I2)

One notebook for the evaluation of every trained detector (docs/PROTOCOL.md: Part A, section 2; extension detectors
RT-DETR-l and Faster R-CNN, section 11.3).

**Two modes, set in the first code cell:**
- `OPEN_TEST = False` (**dry run**, default): validation split only. Checks the whole pipeline on Kaggle
  with the detectors that exist. Safe to run any time; the test split is never read.
- `OPEN_TEST = True` (**a test session**): only when every detector in `RUNS` has FINISHED, its best.pt is frozen and the
  code is tagged. Runs predictions on valid and test (I1), the RQ1 premise test on the test pairs (I3), standard
  validation on test (I4) and the patient-level analysis (I2). Each session opens the test split once (Part A: done
  2026-11-21; RD_s0 + FR_s0: one session of their own):
  if a bug is found afterwards, it is fixed, everything is rerun, and the event is logged (protocol section 2).

Settings: GPU **T4 x1** is enough (about 1-1.5 h for the 8 YOLO detectors, less for RD + FR), Internet **on**.
Inputs: the images, `vgra-index`, and ONE dataset with the frozen runs (`runs/<name>/weights/best.pt`): Part A
`vgra-final-detectors`, RD + FR `vgra-rd-fr-s0`. Every best.pt must match the SHA-256 in `docs/final_weights.json`.
For RD and FR, I3 reads the I1 tables (`--from-pred`, protocol section 10) instead of running the detector again.
The output of this notebook (`/kaggle/working/runs/...`) holds every table; download it or make it a dataset.
Every step runs through `sh()`, which stops the notebook if the step fails, so no step runs on missing files.

In [ ]:
# Settings
REPO = "https://github.com/nithoukhan1/vgra-yolo.git"
REF = "main"        # dry run: main is fine; a test session: a fixed tag made just before ("parta-test-v1", "ext-test-v1")
OPEN_TEST = False   # True ONLY for a test session
RUNS = ["RD_s0", "FR_s0"]
# Test sessions, each opening the test split once (attach ONE dataset with the frozen runs; every best.pt must match
# docs/final_weights.json):
#   Part A (done 2026-11-21, dataset vgra-final-detectors): ["B0_s0", "B0_s1", "B0_s2", "B1_s0", "V8_s0", "V8_s1", "V8R_s0", "V8R_s1"]
#   extension detectors (protocol 11.3, dataset vgra-rd-fr-s0): ["RD_s0", "FR_s0"]
RUN_ARGS = " ".join(RUNS)
READY = False       # set True by the check cell; every later cell refuses to run without it

In [ ]:
# Clone the fixed code, install, check
%cd /kaggle/working
!rm -rf vgra-yolo
!git clone --quiet {REPO} vgra-yolo
%cd /kaggle/working/vgra-yolo
!git checkout --quiet {REF}
!git log -1 --oneline
import subprocess
head = subprocess.run(["git", "rev-parse", "HEAD"], capture_output=True, text=True).stdout.strip()
ref = subprocess.run(["git", "rev-parse", "--verify", "--quiet", f"{REF}^{{commit}}"], capture_output=True, text=True).stdout.strip()
if not head or head != ref:  # e.g. the tag is not on GitHub yet: stop before anything runs
    raise RuntimeError(f"STOP: the code is not at {REF}. Is {REF} pushed to GitHub? Nothing has run.")
print(f"code at {REF}: {head[:7]}")
!pip install -q -r requirements.txt
!pip install -q -e .
!python scripts/00_check_setup.py

In [ ]:
# Every run must be found (and FINISHED for the test session) before anything is evaluated
import hashlib
import json
import subprocess
import sys
sys.path.insert(0, "/kaggle/working/vgra-yolo")
from vgra.runs import find_previous_run, is_finished, read_metrics


def sh(cmd):
    """Run one step, show its output, and stop the notebook if it fails."""
    print(f"$ {cmd}", flush=True)
    proc = subprocess.Popen(cmd, shell=True, cwd="/kaggle/working/vgra-yolo", stdout=subprocess.PIPE,
                            stderr=subprocess.STDOUT, text=True, bufsize=1)
    for line in proc.stdout:
        print(line, end="", flush=True)
    if proc.wait() != 0:
        raise RuntimeError(f"step failed (exit code {proc.returncode}): {cmd}")


FROZEN = json.load(open("/kaggle/working/vgra-yolo/docs/final_weights.json"))  # SHA-256 of the frozen best.pt files


def sha256(path):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for block in iter(lambda: f.read(1 << 20), b""):
            h.update(block)
    return h.hexdigest()


problems, warnings = [], []
for r in RUNS:
    p = find_previous_run(r, "/kaggle/input")
    if p is None or not (p / "weights" / "best.pt").is_file():
        problems.append(f"{r}: not found under /kaggle/input (attach its dataset)")
        continue
    v = read_metrics(p).get("valid", {})
    print(f"{r}: {p} | finished {is_finished(p)} | valid mAP50 {v.get('map50')} | mAP50-95 {v.get('map50_95')}")
    if not is_finished(p):
        (problems if OPEN_TEST else warnings).append(f"{r}: not finished")
    digest = sha256(p / "weights" / "best.pt")
    frozen = FROZEN.get(r, {}).get("sha256")
    print(f"    best.pt sha256 {digest[:16]}... " + ("matches docs/final_weights.json" if digest == frozen
          else "DOES NOT MATCH docs/final_weights.json" if frozen else "not in docs/final_weights.json"))
    if digest != frozen:
        (problems if OPEN_TEST else warnings).append(f"{r}: best.pt is not the frozen file")
if problems:
    raise RuntimeError("STOP: " + "; ".join(problems))  # a real error, so Run All stops here
for w in warnings:
    print(f"WARNING (dry run only): {w}; the test session needs the finished run")
from vgra import utils as vgra_utils
KINDS = {r: vgra_utils.load_config(f"/kaggle/working/vgra-yolo/configs/{r.rsplit('_s', 1)[0]}.yaml")["model"].get("kind", "yolo")
         for r in RUNS}
I3_FROM_PRED = any(k != "yolo" for k in KINDS.values())  # RD, FR: I3 reads the I1 tables (protocol section 10)
print("detector kinds:", KINDS, "| I3 from the I1 tables:", I3_FROM_PRED)
print("\nOPEN_TEST is True: this session opens the test split (protocol section 2)." if OPEN_TEST
      else "\nDry run: validation split only.")
READY = True

In [ ]:
# I1: predictions on valid (every run), then on test (test session only)
assert READY, "the check cell did not pass; fix it and run again"
sh(f"python scripts/10_predict.py --runs {RUN_ARGS} --split valid")
if OPEN_TEST:
    sh(f"python scripts/10_predict.py --runs {RUN_ARGS} --split test --open-test")

In [ ]:
# I3: RQ1 premise test on the test pairs (test session only; on valid it was Gate 2, already done).
# Dry run: a short version on 64 valid pairs, only to prove the step works. RD and FR: from the I1 tables.
assert READY, "the check cell did not pass; fix it and run again"
FROM = " --from-pred /kaggle/working/runs/predictions" if I3_FROM_PRED else ""
if OPEN_TEST:
    sh(f"python scripts/04_premise_test.py --runs {RUN_ARGS} --split test --open-test{FROM}")
else:
    sh(f"python scripts/04_premise_test.py --runs {RUN_ARGS} --split valid --limit 64 --boot 50 --out /kaggle/working/dry{FROM}")

In [ ]:
# I4: standard Ultralytics validation (detector table T2)
assert READY, "the check cell did not pass; fix it and run again"
split = "test --open-test" if OPEN_TEST else "valid"
sh(f"python scripts/11_val_detectors.py --runs {RUN_ARGS} --split {split}")

In [ ]:
# I2: patient-level analysis (CPU), thresholds from valid
assert READY, "the check cell did not pass; fix it and run again"
split = "test" if OPEN_TEST else "valid"
sh(f"python scripts/12_study_level.py --pred /kaggle/working/runs/predictions --runs {RUN_ARGS} --eval {split}")

In [ ]:
# What was produced
sh(r'find /kaggle/working/runs -maxdepth 4 \( -name "*.csv" -o -name "*.json" -o -name "summary.txt" \) -not -path "*/lists/*" | sort')